# Modify Models #

This JupyterNotebook file will modify the desired models with the desired pathways. It is designed to add the desired substrate assimilation pathways to the product models, but could be used to add other pathways to models as well.

**Functions:**

1 - modifyModels(inputDirectory_createModels, outputDirectory_createModels)

2 - addAssimilationPWs(model, inputDirectory_editModels)

**Requirements:**
- Models to be edited must be.xml files and have the prefix "PRE" added to the file name (ie. PREmodel_GS_Farnesene.xml); the PRE prefix will be removed in the exported files. Code could be modified to change this requirement if desired.
- Excel file format be used for adding new pathways/reactions to the model(s).
- When designing the pathways in the Excel file formate, users should ensure that the reaction and metabolite IDs being used are consistent with the BiGG database (the code will check IDs against the model to ensure nothing is added in duplicate, but this relies on correct naming convention on the user side).

** *See SPI Analysis Log at bottom of document*

In [ ]:
import cobra
import cameo
import math
import escher
import plotly
import os

import numpy as np
from scipy import stats
import pandas as pd
import sympy as sy
from datetime import date, datetime
import time
import glob #For createModels() function

date = datetime.strftime(datetime.now(), '%Y-%m-%d')

In [ ]:
## Main - Modifying models ##
#Run other functions below this cell first

#Directory for models to be edited:


inputDirectory_createModels = os.path.abspath("ModifyModels/Input/")

#Directory for model output after editing:
outputDirectory_createModels = os.path.abspath("ModifyModels/Output/")

#Directory for Excel files with pathways to add to models
inputDirectory_editModels = os.path.abspath("ModifyModels/Input/EditModels/") #Currently global var

tic = time.perf_counter() #Timer start
#Add the pathways to the models and export them to desired folder:
modifyModels(inputDirectory_createModels, outputDirectory_createModels)
toc = time.perf_counter()
print(f"Models modified in {toc - tic:0.4f} seconds")

In [ ]:
def modifyModels(inputDirectory_createModels, outputDirectory_createModels):
    for files in glob.glob1(inputDirectory_createModels, '*.xml'):
        #Load model
        modelName = files.replace(inputDirectory_createModels, '') #Remove folder path from filenames
        model = cameo.load_model(inputDirectory_createModels + '/' + modelName)
        #display(modelName)#Check
        
        #Edit the model
        addAssimilationPWs(model, inputDirectory_editModels)
        print('Changes made to model:', modelName) #Check
    
        #Save modified model as a new model:
        newModelName = modelName.replace('PRE', '')
        print('New model name', newModelName)
        cobra.io.write_sbml_model(model, outputDirectory_createModels + '\\' + newModelName)
        
        print('Model exported:', newModelName) #Check
        print('\n')
    
    return()

In [ ]:
def addAssimilationPWs(model, inputDirectory_editModels):
    
    #Make lists of all the metabolites in the model and all of the reactions in the model
    met_ids = [metabolite.id for metabolite in model.metabolites]
    rxn_ids = [reaction.id for reaction in model.reactions]
    
    #Loop through files
    for files in glob.glob1(inputDirectory_editModels, '*.xlsx'): 
        #print(files) #Check
        fileNames = files.replace(inputDirectory_editModels, '')
        print('\n',fileNames) #Check
        
        #Read in metabolites
        metabolitesDF = pd.read_excel(inputDirectory_editModels+ '\\' +fileNames, sheet_name='Metabolites')
        #display(metabolitesDF) #Check
        
        #Read in reactions
        reactionsDF = pd.read_excel(inputDirectory_editModels+ '\\' + fileNames, sheet_name='Reactions')
        #display(reactionsDF) #Check
        
        #Loop through all metabolites in metabolitesDF, add any missing from the model
        for index, rows in metabolitesDF.iterrows():
    
            metID = metabolitesDF.loc[index].metID #Assign metabolite ID from dataframe
            if not metID in met_ids: #If metabolite doesn't yet exist in the model, create new metabolite (otherwise no action needed)
                #Assign metabolite fields from dataframe:
                metFormula = metabolitesDF.loc[index].metFormula
                metName = metabolitesDF.loc[index].metName
                metCompartment = metabolitesDF.loc[index].metCompartment
                metCharge = int(metabolitesDF.loc[index].metCharge)
                #print('metCharge is type: ', type(metCharge))#Check
                #display(metID, metFormula, metName, metCompartment, metCharge) #Check
    
                #Create metabolite:
                metabolite = cobra.Metabolite(metID, formula = metFormula, name = metName, compartment = metCompartment, charge = metCharge)
                
                #Add metabolite to the model:
                model.add_metabolites([metabolite])
                #print('Metabolite added:', metabolite)#Check
                #display(metabolite) #Check
                
                #Add metabolite ID to the list of metIDs, so that if future assimilation pathways have the same metabolite,
                #it's not added twice (or doesn't give error message)
                met_ids.append(metID)
    
        #display(model.metabolites) #Check
        #display(model.metabolites.get_by_id('test_e'))#Check
    
        #Loop through all reactions in reactionsDF, add any missing from the model
        for index, rows in reactionsDF.iterrows():
    
            rxnID = reactionsDF.loc[index].rxnID #Assign reaction ID from dataframe
            if not rxnID in rxn_ids: #If reaction doesn't yet exist in model, create and add it
                reaction = cobra.Reaction(reactionsDF.loc[index].rxnID)
                reaction.name = reactionsDF.loc[index].rxnName
                reaction.subsystem = reactionsDF.loc[index].rxnSubsystem
                reaction.lower_bound = reactionsDF.loc[index].rxnLB
                reaction.upper_bound = reactionsDF.loc[index].rxnUB

                #Add the reaction to the model (Note: its reaction field will be empty at first):
                model.add_reactions([reaction])
    
                #Add the reaction stoichiometry to the new reaction's reaction field (supplied from reactionsDF)
                rxn = model.reactions.get_by_id(rxnID)
                rxn.reaction = reactionsDF.loc[index].rxnAddMetabolites
                
                #Add reaction ID to the list of rxnIDs, so that if future assimilation pathways have the same reaction,
                #it's not added twice (or doesn't give error message)
                rxn_ids.append(rxnID)
                
                #Need to set reaction bounds again after adding reaction to model, or it defaults knocked out reactions [0,0] to [0,1000]
                model.reactions.get_by_id(rxnID).lower_bound = float(reactionsDF.loc[index].rxnLB)
                model.reactions.get_by_id(rxnID).upper_bound = float(reactionsDF.loc[index].rxnUB)
            
                print('Reaction added:', rxn) #Check
                #display(rxn) #Check

    return(model)


### Quick Code for Acetyl-CoA Investigation ###

In [ ]:
model =  cameo.models.bigg.iHN637 #iHN637 #iJO1366
modelName = 'model_GS_iHN637_AcetylCoA.xml'#'model_iJN1463.xml'


accoa_c = model.metabolites.get_by_id('accoa_c')
coa_c = model.metabolites.get_by_id('coa_c')
acetyl_c = cobra.Metabolite('acetyl_c', formula = 'C2H3O', name = 'acetyl', compartment = 'c', charge=0)
acetyl_e = cobra.Metabolite('acetyl_e', formula = 'C2H3O', name = 'acetyl', compartment = 'e', charge=0)

#Remove CoA group from acetyl...
reaction = cobra.Reaction('AcCoATest')
reaction.name = 'AcetylCoA Test'
reaction.subsystem = "unknown"
reaction.lower_bound = 0 
reaction.upper_bound = 1000 

reaction.add_metabolites({accoa_c: -1, coa_c:1, acetyl_c:1 })
model.add_reactions([reaction])


#Transport acetyl group
reaction = cobra.Reaction('AcetylTransport')
reaction.name = 'Acetyl transport'
reaction.subsystem = "unknown"
reaction.lower_bound = -1000 #export rate
reaction.upper_bound = 1000 #uptake rate


#Add the metabolites and their stoichiometries to the reaction:
reaction.add_metabolites({acetyl_e: -1, acetyl_c:1})

#Add the reaction to the model:
model.add_reactions([reaction])


#Exchange rxn for acetyl group
reaction = cobra.Reaction('EX_acetyl_e')
reaction.name = 'Acetyl exchange'
reaction.subsystem = "unknown"
reaction.lower_bound = 0 #uptake rate (specifies highest rate at which EG enters system)
reaction.upper_bound = 1000 #export rate (specifies highest rate at which EG can leave the system; ie. be produced)


#Add the metabolites and their stoichiometries to the reaction:
reaction.add_metabolites({acetyl_e: -1})

#Add the reaction to the model:
model.add_reactions([reaction])

#Check model:
display(model)



#Add assimilation pathways
#Directory for Excel files with pathways to add to models
inputDirectory_editModels = 'SP_Investigation/Overall_Analysis_Docs/ModifyModels/Input/EditModels/' #Currently global var
#modelB = addAssimilationPWs(model, inputDirectory_editModels)
model.reactions.EX_fru_e.lower_bound=0
display(model.reactions.EX_fru_e)
modelB=model

#Check model:
print('Model after adding assimilation pathways:')
display(modelB)
display(modelB.reactions.ATPM)
display(modelB.reactions.EX_acetyl_e)


cobra.io.write_sbml_model(modelB, 'SP_Investigation/Overall_Analysis_Docs/ModifyModels/Output/' + modelName)